In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed_rework_v2"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "rework_v2"
    / "03_adak_calculation"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RQ1_PATH = (
    DATA_DIR
    / "02_RQ1_Complete_Triplets_Long.csv"
)

RQ2_PATH = (
    DATA_DIR
    / "02_RQ2_Balanced_Complete_Triplets_Long.csv"
)

RQ3_PATH = (
    DATA_DIR
    / "02_RQ3_Original_Methods.csv"
)

SOURCE_MANIFEST_PATH = (
    DATA_DIR
    / "02_Distractor_Generation_Final_Manifest.json"
)

DATASET_SUMMARY_PATH = (
    OUTPUT_DIR
    / "03_input_dataset_summary.csv"
)

LINE_AUDIT_PATH = (
    OUTPUT_DIR
    / "03_comment_line_preservation_audit.csv"
)

LINE_SUMMARY_PATH = (
    OUTPUT_DIR
    / "03_comment_line_preservation_summary.csv"
)

INPUT_MANIFEST_PATH = (
    OUTPUT_DIR
    / "03_input_audit_manifest.json"
)


def count_physical_lines(text):
    return max(
        len(str(text).splitlines()),
        1,
    )


def count_nonempty_lines(text):
    cleaned = str(text).strip()

    return max(
        sum(
            bool(line.strip())
            for line in cleaned.splitlines()
        ),
        1,
    )


for path in [
    RQ1_PATH,
    RQ2_PATH,
    RQ3_PATH,
    SOURCE_MANIFEST_PATH,
]:
    assert path.exists(), path

rq1 = pd.read_csv(
    RQ1_PATH,
    keep_default_na=False,
)

rq2 = pd.read_csv(
    RQ2_PATH,
    keep_default_na=False,
)

rq3 = pd.read_csv(
    RQ3_PATH,
    keep_default_na=False,
)

with SOURCE_MANIFEST_PATH.open(
    mode="r",
    encoding="utf-8",
) as handle:
    source_manifest = json.load(handle)


required_columns = {
    "sample_id",
    "family_id",
    "language",
    "repository",
    "condition",
    "code",
    "comment",
    "comment_line_count",
}

for dataset_name, dataset in {
    "RQ1": rq1,
    "RQ2": rq2,
    "RQ3": rq3,
}.items():
    missing_columns = (
        required_columns
        - set(dataset.columns)
    )

    assert not missing_columns, (
        f"{dataset_name} missing columns: "
        f"{sorted(missing_columns)}"
    )

    assert dataset[
        "family_id"
    ].str.strip().ne("").all()

    assert dataset[
        "code"
    ].str.strip().ne("").all()

    assert dataset[
        "comment"
    ].str.strip().ne("").all()


expected_conditions = {
    "original",
    "topic_swap",
    "grammar_shuffle",
}

assert set(rq1["condition"]) == expected_conditions
assert set(rq2["condition"]) == expected_conditions
assert set(rq3["condition"]) == {"original"}

assert not rq1.duplicated(
    [
        "family_id",
        "condition",
    ]
).any()

assert not rq2.duplicated(
    [
        "family_id",
        "condition",
    ]
).any()

assert rq3["family_id"].is_unique

assert (
    rq1.groupby(
        "family_id"
    )["condition"].nunique()
    == 3
).all()

assert (
    rq2.groupby(
        "family_id"
    )["condition"].nunique()
    == 3
).all()

assert (
    rq1.groupby(
        "family_id"
    )["code"].nunique()
    == 1
).all()

assert (
    rq2.groupby(
        "family_id"
    )["code"].nunique()
    == 1
).all()

assert set(
    rq2["family_id"]
).issubset(
    set(rq1["family_id"])
)

assert set(
    rq1["family_id"]
).issubset(
    set(rq3["family_id"])
)

assert len(rq1) == int(
    source_manifest["rq1"]["rows"]
)

assert rq1["family_id"].nunique() == int(
    source_manifest["rq1"]["families"]
)

assert len(rq2) == int(
    source_manifest["rq2"]["rows"]
)

assert rq2["family_id"].nunique() == int(
    source_manifest["rq2"][
        "total_families"
    ]
)

assert rq3["family_id"].nunique() == int(
    source_manifest["rq3"][
        "total_families"
    ]
)


dataset_summary = pd.DataFrame(
    [
        {
            "dataset": "RQ1",
            "rows": len(rq1),
            "families": (
                rq1["family_id"].nunique()
            ),
            "languages": (
                rq1["language"].nunique()
            ),
            "conditions": (
                rq1["condition"].nunique()
            ),
        },
        {
            "dataset": "RQ2",
            "rows": len(rq2),
            "families": (
                rq2["family_id"].nunique()
            ),
            "languages": (
                rq2["language"].nunique()
            ),
            "conditions": (
                rq2["condition"].nunique()
            ),
        },
        {
            "dataset": "RQ3",
            "rows": len(rq3),
            "families": (
                rq3["family_id"].nunique()
            ),
            "languages": (
                rq3["language"].nunique()
            ),
            "conditions": (
                rq3["condition"].nunique()
            ),
        },
    ]
)


line_audit = rq1[
    [
        "sample_id",
        "family_id",
        "language",
        "repository",
        "condition",
        "comment_line_count",
        "comment",
    ]
].copy()

line_audit[
    "physical_line_count"
] = line_audit[
    "comment"
].map(
    count_physical_lines
)

line_audit[
    "nonempty_line_count"
] = line_audit[
    "comment"
].map(
    count_nonempty_lines
)


original_line_counts = (
    line_audit.loc[
        line_audit[
            "condition"
        ] == "original",
        [
            "family_id",
            "physical_line_count",
            "nonempty_line_count",
        ],
    ]
    .rename(
        columns={
            "physical_line_count": (
                "original_physical_line_count"
            ),
            "nonempty_line_count": (
                "original_nonempty_line_count"
            ),
        }
    )
)

line_audit = line_audit.merge(
    original_line_counts,
    on="family_id",
    how="left",
    validate="many_to_one",
)

line_audit[
    "stored_matches_physical"
] = (
    line_audit[
        "comment_line_count"
    ]
    == line_audit[
        "physical_line_count"
    ]
)

line_audit[
    "physical_lines_preserved"
] = (
    line_audit[
        "physical_line_count"
    ]
    == line_audit[
        "original_physical_line_count"
    ]
)

line_audit[
    "nonempty_lines_preserved"
] = (
    line_audit[
        "nonempty_line_count"
    ]
    == line_audit[
        "original_nonempty_line_count"
    ]
)

line_audit[
    "nonempty_line_difference"
] = (
    line_audit[
        "nonempty_line_count"
    ]
    - line_audit[
        "original_nonempty_line_count"
    ]
)

line_audit[
    "absolute_nonempty_line_difference"
] = line_audit[
    "nonempty_line_difference"
].abs()


line_summary = (
    line_audit
    .groupby(
        [
            "language",
            "condition",
        ],
        as_index=False,
    )
    .agg(
        records=(
            "family_id",
            "count",
        ),
        stored_matches_physical=(
            "stored_matches_physical",
            "sum",
        ),
        physical_lines_preserved=(
            "physical_lines_preserved",
            "sum",
        ),
        nonempty_lines_preserved=(
            "nonempty_lines_preserved",
            "sum",
        ),
        mean_absolute_nonempty_difference=(
            "absolute_nonempty_line_difference",
            "mean",
        ),
        maximum_absolute_nonempty_difference=(
            "absolute_nonempty_line_difference",
            "max",
        ),
    )
)

for column in [
    "stored_matches_physical",
    "physical_lines_preserved",
    "nonempty_lines_preserved",
]:
    line_summary[
        f"{column}_percent"
    ] = (
        100
        * line_summary[column]
        / line_summary["records"]
    ).round(2)

line_summary[
    "mean_absolute_nonempty_difference"
] = line_summary[
    "mean_absolute_nonempty_difference"
].round(3)


dataset_summary.to_csv(
    DATASET_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)

line_audit.to_csv(
    LINE_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

line_summary.to_csv(
    LINE_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)


input_manifest = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_manifest": str(
        SOURCE_MANIFEST_PATH
    ),
    "rq1_path": str(RQ1_PATH),
    "rq2_path": str(RQ2_PATH),
    "rq3_path": str(RQ3_PATH),
    "old_notebook_comment_line_rule": (
        "Non-empty lines after stripping "
        "leading and trailing whitespace, "
        "with a minimum of one line."
    ),
    "stored_comment_line_rule": (
        "Physical splitlines count, "
        "with a minimum of one line."
    ),
    "outputs": {
        "dataset_summary": str(
            DATASET_SUMMARY_PATH
        ),
        "line_audit": str(
            LINE_AUDIT_PATH
        ),
        "line_summary": str(
            LINE_SUMMARY_PATH
        ),
    },
}

with INPUT_MANIFEST_PATH.open(
    mode="w",
    encoding="utf-8",
) as handle:
    json.dump(
        input_manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


print("Input dataset summary")
display(dataset_summary)

print("\nComment-line preservation summary")
display(
    line_summary[
        [
            "language",
            "condition",
            "records",
            "stored_matches_physical_percent",
            "physical_lines_preserved_percent",
            "nonempty_lines_preserved_percent",
            "mean_absolute_nonempty_difference",
            "maximum_absolute_nonempty_difference",
        ]
    ]
)

print("\nSaved outputs")
print("Dataset summary:", DATASET_SUMMARY_PATH)
print("Line audit:", LINE_AUDIT_PATH)
print("Line summary:", LINE_SUMMARY_PATH)
print("Manifest:", INPUT_MANIFEST_PATH)

Input dataset summary


,dataset,rows,families,languages,conditions
0,RQ1,20910,6970,6,3
1,RQ2,20196,6732,6,3
2,RQ3,7500,7500,6,1



Comment-line preservation summary


,language,condition,records,stored_matches_physical_percent,physical_lines_preserved_percent,nonempty_lines_preserved_percent,mean_absolute_nonempty_difference,maximum_absolute_nonempty_difference
0,go,grammar_shuffle,1209,100.0,100.0,100.00,0.000,0
1,go,original,1209,100.0,100.0,100.00,0.000,0
2,go,topic_swap,1209,100.0,100.0,99.92,0.001,1
3,java,grammar_shuffle,1146,100.0,100.0,100.00,0.000,0
4,java,original,1146,100.0,100.0,100.00,0.000,0
5,java,topic_swap,1146,100.0,100.0,70.68,0.329,6
6,javascript,grammar_shuffle,1141,100.0,100.0,100.00,0.000,0
7,javascript,original,1141,100.0,100.0,100.00,0.000,0
8,javascript,topic_swap,1141,100.0,100.0,76.95,0.280,4
9,php,grammar_shuffle,1122,100.0,100.0,100.00,0.000,0



Saved outputs
Dataset summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\03_adak_calculation\03_input_dataset_summary.csv
Line audit: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\03_adak_calculation\03_comment_line_preservation_audit.csv
Line summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\03_adak_calculation\03_comment_line_preservation_summary.csv
Manifest: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\03_adak_calculation\03_input_audit_manifest.json


In [2]:
schema_rows = []

for dataset_name, dataset in {
    "RQ1": rq1,
    "RQ2": rq2,
    "RQ3": rq3,
}.items():
    for column in dataset.columns:
        values = dataset[column]

        schema_rows.append(
            {
                "dataset": dataset_name,
                "column": column,
                "dtype": str(values.dtype),
                "records": len(values),
                "missing_values": int(
                    values.isna().sum()
                ),
                "empty_strings": int(
                    values.astype(str)
                    .str.strip()
                    .eq("")
                    .sum()
                ),
                "unique_values": int(
                    values.nunique(
                        dropna=False
                    )
                ),
            }
        )

schema_audit = pd.DataFrame(
    schema_rows
)

token_columns = sorted(
    {
        column
        for dataset in [
            rq1,
            rq2,
            rq3,
        ]
        for column in dataset.columns
        if "token" in column.casefold()
    }
)

method_columns = sorted(
    {
        column
        for dataset in [
            rq1,
            rq2,
            rq3,
        ]
        for column in dataset.columns
        if "method" in column.casefold()
        or "function" in column.casefold()
    }
)

complexity_columns = sorted(
    {
        column
        for dataset in [
            rq1,
            rq2,
            rq3,
        ]
        for column in dataset.columns
        if any(
            term in column.casefold()
            for term in [
                "complex",
                "cyclomatic",
                "nloc",
                "loc",
                "parameter",
            ]
        )
    }
)


token_audit_rows = []

for dataset_name, dataset in {
    "RQ1": rq1,
    "RQ2": rq2,
    "RQ3": rq3,
}.items():
    for column in token_columns:
        if column not in dataset.columns:
            continue

        values = dataset[column]

        numeric_values = pd.to_numeric(
            values,
            errors="coerce",
        )

        nonempty_mask = (
            values.astype(str)
            .str.strip()
            .ne("")
        )

        numeric_mask = (
            numeric_values.notna()
        )

        token_audit_rows.append(
            {
                "dataset": dataset_name,
                "column": column,
                "dtype": str(
                    values.dtype
                ),
                "records": len(values),
                "nonempty_records": int(
                    nonempty_mask.sum()
                ),
                "numeric_records": int(
                    numeric_mask.sum()
                ),
                "numeric_percent": round(
                    100
                    * numeric_mask.mean(),
                    2,
                ),
                "minimum_numeric_value": (
                    float(
                        numeric_values.min()
                    )
                    if numeric_mask.any()
                    else None
                ),
                "maximum_numeric_value": (
                    float(
                        numeric_values.max()
                    )
                    if numeric_mask.any()
                    else None
                ),
                "nonpositive_numeric_values": int(
                    numeric_values.le(
                        0
                    ).sum()
                ),
                "unique_values": int(
                    values.nunique(
                        dropna=False
                    )
                ),
            }
        )

token_audit = pd.DataFrame(
    token_audit_rows
)


family_consistency_rows = []

for dataset_name, dataset in {
    "RQ1": rq1,
    "RQ2": rq2,
}.items():
    for column in token_columns:
        if column not in dataset.columns:
            continue

        inconsistent_families = int(
            dataset.groupby(
                "family_id"
            )[column]
            .nunique(
                dropna=False
            )
            .gt(1)
            .sum()
        )

        family_consistency_rows.append(
            {
                "dataset": dataset_name,
                "column": column,
                "families": int(
                    dataset[
                        "family_id"
                    ].nunique()
                ),
                "inconsistent_families": (
                    inconsistent_families
                ),
            }
        )

family_token_consistency = pd.DataFrame(
    family_consistency_rows
)


SCHEMA_AUDIT_PATH = (
    OUTPUT_DIR
    / "03_input_schema_audit.csv"
)

TOKEN_AUDIT_PATH = (
    OUTPUT_DIR
    / "03_token_column_audit.csv"
)

TOKEN_CONSISTENCY_PATH = (
    OUTPUT_DIR
    / "03_token_family_consistency.csv"
)

schema_audit.to_csv(
    SCHEMA_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

token_audit.to_csv(
    TOKEN_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

family_token_consistency.to_csv(
    TOKEN_CONSISTENCY_PATH,
    index=False,
    encoding="utf-8",
)


print("RQ1 columns")
print(rq1.columns.tolist())

print("\nRQ2 columns")
print(rq2.columns.tolist())

print("\nRQ3 columns")
print(rq3.columns.tolist())

print("\nToken-related columns")
print(token_columns)

print("\nMethod-related columns")
print(method_columns)

print("\nComplexity-related columns")
print(complexity_columns)

print("\nToken-column audit")
display(token_audit)

print("\nFamily-level token consistency")
display(family_token_consistency)

RQ1 columns
['sample_id', 'family_id', 'language', 'repository', 'sample_order', 'within_repository_rank', 'original_split', 'file_path', 'function_name', 'commit_sha', 'source_url', 'shard_relative_path', 'shard_line_number', 'code_token_count', 'comment_token_count', 'combined_token_count', 'code_line_count', 'comment_line_count', 'exact_code_hash', 'exact_comment_hash', 'exact_pair_hash', 'condition', 'code', 'comment', 'pair_token_count', 'donor_family_id', 'donor_sample_id', 'donor_repository', 'unigram_jaccard', 'bigram_jaccard', 'fixed_position_fraction']

RQ2 columns
['sample_id', 'family_id', 'language', 'repository', 'sample_order', 'within_repository_rank', 'original_split', 'file_path', 'function_name', 'commit_sha', 'source_url', 'shard_relative_path', 'shard_line_number', 'code_token_count', 'comment_token_count', 'combined_token_count', 'code_line_count', 'comment_line_count', 'exact_code_hash', 'exact_comment_hash', 'exact_pair_hash', 'condition', 'code', 'comment', 'pa

,dataset,column,dtype,records,nonempty_records,numeric_records,numeric_percent,minimum_numeric_value,maximum_numeric_value,nonpositive_numeric_values,unique_values
0,RQ1,code_token_count,int64,20910,20910,20910,100.0,15.0,368.0,0,252
1,RQ1,combined_token_count,int64,20910,20910,20910,100.0,25.0,386.0,0,269
2,RQ1,comment_token_count,int64,20910,20910,20910,100.0,1.0,176.0,0,106
3,RQ1,pair_token_count,int64,20910,20910,20910,100.0,37.0,512.0,0,476
4,RQ2,code_token_count,int64,20196,20196,20196,100.0,15.0,368.0,0,248
5,RQ2,combined_token_count,int64,20196,20196,20196,100.0,25.0,386.0,0,265
6,RQ2,comment_token_count,int64,20196,20196,20196,100.0,1.0,176.0,0,106
7,RQ2,pair_token_count,int64,20196,20196,20196,100.0,37.0,512.0,0,476
8,RQ3,code_token_count,int64,7500,7500,7500,100.0,15.0,368.0,0,253
9,RQ3,combined_token_count,int64,7500,7500,7500,100.0,25.0,386.0,0,276



Family-level token consistency


,dataset,column,families,inconsistent_families
0,RQ1,code_token_count,6970,0
1,RQ1,combined_token_count,6970,0
2,RQ1,comment_token_count,6970,0
3,RQ1,pair_token_count,6970,6544
4,RQ2,code_token_count,6732,0
5,RQ2,combined_token_count,6732,0
6,RQ2,comment_token_count,6732,0
7,RQ2,pair_token_count,6732,6321


In [3]:
FAMILY_METRICS_PATH = (
    OUTPUT_DIR
    / "03_family_code_measurements.csv"
)

RQ1_ADAK_PATH = (
    DATA_DIR
    / "03_RQ1_Adak_Enriched.csv"
)

RQ2_ADAK_PATH = (
    DATA_DIR
    / "03_RQ2_Adak_Enriched.csv"
)

RQ3_ADAK_PATH = (
    DATA_DIR
    / "03_RQ3_Adak_Enriched.csv"
)

ADAK_SUMMARY_PATH = (
    OUTPUT_DIR
    / "03_adak_summary.csv"
)

ADAK_INVARIANCE_PATH = (
    OUTPUT_DIR
    / "03_adak_invariance_audit.csv"
)


family_metrics = rq3[
    [
        "family_id",
        "language",
        "repository",
        "code_token_count",
        "code_line_count",
        "exact_code_hash",
    ]
].copy()

assert family_metrics[
    "family_id"
].is_unique

assert family_metrics[
    "code_token_count"
].gt(0).all()

assert family_metrics[
    "code_line_count"
].gt(0).all()

family_metrics[
    "method_count"
] = 1

family_metrics[
    "sfv"
] = (
    family_metrics[
        "code_token_count"
    ]
    / family_metrics[
        "method_count"
    ]
    * 0.3
)

family_metrics[
    "code_measurement_status"
] = "valid"

assert family_metrics[
    "sfv"
].gt(0).all()


family_code_token_map = (
    family_metrics.set_index(
        "family_id"
    )["code_token_count"]
)

family_sfv_map = (
    family_metrics.set_index(
        "family_id"
    )["sfv"]
)

family_method_count_map = (
    family_metrics.set_index(
        "family_id"
    )["method_count"]
)


for dataset_name, dataset in {
    "RQ1": rq1,
    "RQ2": rq2,
    "RQ3": rq3,
}.items():
    expected_code_tokens = (
        dataset[
            "family_id"
        ].map(
            family_code_token_map
        )
    )

    assert expected_code_tokens.notna().all()

    assert (
        dataset[
            "code_token_count"
        ]
        == expected_code_tokens
    ).all(), (
        f"{dataset_name} contains code-token "
        "values inconsistent with RQ3."
    )


def add_original_adak_metrics(
    dataset,
):
    enriched = dataset.copy()

    enriched[
        "method_count"
    ] = enriched[
        "family_id"
    ].map(
        family_method_count_map
    )

    enriched[
        "comment_lines_physical"
    ] = enriched[
        "comment"
    ].map(
        count_physical_lines
    )

    enriched[
        "comment_lines_nonempty"
    ] = enriched[
        "comment"
    ].map(
        count_nonempty_lines
    )

    enriched[
        "stored_comment_lines_match_physical"
    ] = (
        enriched[
            "comment_line_count"
        ]
        == enriched[
            "comment_lines_physical"
        ]
    )

    enriched[
        "mcv"
    ] = (
        enriched[
            "comment_lines_physical"
        ]
        / enriched[
            "method_count"
        ]
        * 0.8
    )

    enriched[
        "sfv"
    ] = enriched[
        "family_id"
    ].map(
        family_sfv_map
    )

    enriched[
        "adak_index"
    ] = (
        (
            100
            * enriched[
                "mcv"
            ]
            / enriched[
                "sfv"
            ]
        )
        - 100
    )

    enriched[
        "adak_below_stated_lower_bound"
    ] = (
        enriched[
            "adak_index"
        ]
        < -100
    )

    enriched[
        "adak_above_stated_upper_bound"
    ] = (
        enriched[
            "adak_index"
        ]
        > 100
    )

    enriched[
        "adak_measurement_status"
    ] = "valid"

    assert enriched[
        "method_count"
    ].eq(1).all()

    assert enriched[
        "stored_comment_lines_match_physical"
    ].all()

    assert enriched[
        "mcv"
    ].gt(0).all()

    assert enriched[
        "sfv"
    ].gt(0).all()

    assert enriched[
        "adak_index"
    ].notna().all()

    return enriched


rq1_adak = add_original_adak_metrics(
    rq1
)

rq2_adak = add_original_adak_metrics(
    rq2
)

rq3_adak = add_original_adak_metrics(
    rq3
)


invariance_rows = []

for dataset_name, dataset in {
    "RQ1": rq1_adak,
    "RQ2": rq2_adak,
}.items():
    grouped = dataset.groupby(
        "family_id"
    )

    for metric in [
        "code_token_count",
        "comment_lines_physical",
        "method_count",
        "mcv",
        "sfv",
        "adak_index",
    ]:
        inconsistent_families = int(
            grouped[
                metric
            ]
            .nunique(
                dropna=False
            )
            .gt(1)
            .sum()
        )

        invariance_rows.append(
            {
                "dataset": dataset_name,
                "metric": metric,
                "families": int(
                    dataset[
                        "family_id"
                    ].nunique()
                ),
                "inconsistent_families": (
                    inconsistent_families
                ),
                "invariant_families": int(
                    dataset[
                        "family_id"
                    ].nunique()
                )
                - inconsistent_families,
                "invariant_percent": round(
                    100
                    * (
                        1
                        - (
                            inconsistent_families
                            / dataset[
                                "family_id"
                            ].nunique()
                        )
                    ),
                    2,
                ),
            }
        )


adak_invariance = pd.DataFrame(
    invariance_rows
)

assert (
    adak_invariance[
        "inconsistent_families"
    ]
    == 0
).all()


summary_frames = []

for dataset_name, dataset in {
    "RQ1": rq1_adak,
    "RQ2": rq2_adak,
    "RQ3": rq3_adak,
}.items():
    summary = (
        dataset
        .groupby(
            [
                "language",
                "condition",
            ],
            as_index=False,
        )
        .agg(
            records=(
                "family_id",
                "count",
            ),
            families=(
                "family_id",
                "nunique",
            ),
            minimum_adak=(
                "adak_index",
                "min",
            ),
            median_adak=(
                "adak_index",
                "median",
            ),
            mean_adak=(
                "adak_index",
                "mean",
            ),
            maximum_adak=(
                "adak_index",
                "max",
            ),
            below_stated_lower_bound=(
                "adak_below_stated_lower_bound",
                "sum",
            ),
            above_stated_upper_bound=(
                "adak_above_stated_upper_bound",
                "sum",
            ),
        )
    )

    summary.insert(
        0,
        "dataset",
        dataset_name,
    )

    summary_frames.append(
        summary
    )


adak_summary = pd.concat(
    summary_frames,
    ignore_index=True,
)

for column in [
    "minimum_adak",
    "median_adak",
    "mean_adak",
    "maximum_adak",
]:
    adak_summary[
        column
    ] = adak_summary[
        column
    ].round(4)


family_metrics.to_csv(
    FAMILY_METRICS_PATH,
    index=False,
    encoding="utf-8",
)

rq1_adak.to_csv(
    RQ1_ADAK_PATH,
    index=False,
    encoding="utf-8",
)

rq2_adak.to_csv(
    RQ2_ADAK_PATH,
    index=False,
    encoding="utf-8",
)

rq3_adak.to_csv(
    RQ3_ADAK_PATH,
    index=False,
    encoding="utf-8",
)

adak_summary.to_csv(
    ADAK_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)

adak_invariance.to_csv(
    ADAK_INVARIANCE_PATH,
    index=False,
    encoding="utf-8",
)


print("Adak invariance audit")
display(adak_invariance)

print("\nAdak summary")
display(adak_summary)

print("\nSaved datasets")
print(RQ1_ADAK_PATH)
print(RQ2_ADAK_PATH)
print(RQ3_ADAK_PATH)
print(FAMILY_METRICS_PATH)

Adak invariance audit


,dataset,metric,families,inconsistent_families,invariant_families,invariant_percent
0,RQ1,code_token_count,6970,0,6970,100.0
1,RQ1,comment_lines_physical,6970,0,6970,100.0
2,RQ1,method_count,6970,0,6970,100.0
3,RQ1,mcv,6970,0,6970,100.0
4,RQ1,sfv,6970,0,6970,100.0
5,RQ1,adak_index,6970,0,6970,100.0
6,RQ2,code_token_count,6732,0,6732,100.0
7,RQ2,comment_lines_physical,6732,0,6732,100.0
8,RQ2,method_count,6732,0,6732,100.0
9,RQ2,mcv,6732,0,6732,100.0



Adak summary


,dataset,language,condition,records,families,minimum_adak,median_adak,mean_adak,maximum_adak,below_stated_lower_bound,above_stated_upper_bound
0,RQ1,go,grammar_shuffle,1209,1209,-99.2572,-93.6508,-91.4662,-16.6667,0,0
1,RQ1,go,original,1209,1209,-99.2572,-93.6508,-91.4662,-16.6667,0,0
2,RQ1,go,topic_swap,1209,1209,-99.2572,-93.6508,-91.4662,-16.6667,0,0
3,RQ1,java,grammar_shuffle,1146,1146,-99.2754,-82.9787,-76.0796,98.2906,0,0
4,RQ1,java,original,1146,1146,-99.2754,-82.9787,-76.0796,98.2906,0,0
5,RQ1,java,topic_swap,1146,1146,-99.2754,-82.9787,-76.0796,98.2906,0,0
6,RQ1,javascript,grammar_shuffle,1141,1141,-99.0899,-93.1183,-87.5708,53.8462,0,0
7,RQ1,javascript,original,1141,1141,-99.0899,-93.1183,-87.5708,53.8462,0,0
8,RQ1,javascript,topic_swap,1141,1141,-99.0899,-93.1183,-87.5708,53.8462,0,0
9,RQ1,php,grammar_shuffle,1122,1122,-98.9624,-83.7540,-81.2093,49.5935,0,0



Saved datasets
C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\03_RQ1_Adak_Enriched.csv
C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\03_RQ2_Adak_Enriched.csv
C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\03_RQ3_Adak_Enriched.csv
C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\03_adak_calculation\03_family_code_measurements.csv


In [6]:
import numpy as np


CROSS_DATASET_AUDIT_PATH = (
    OUTPUT_DIR
    / "03_cross_dataset_consistency_audit.csv"
)

FORMULA_AUDIT_PATH = (
    OUTPUT_DIR
    / "03_formula_equivalence_audit.csv"
)

RANGE_AUDIT_PATH = (
    OUTPUT_DIR
    / "03_stated_range_violation_records.csv"
)

RANGE_SUMMARY_PATH = (
    OUTPUT_DIR
    / "03_stated_range_violation_summary.csv"
)

CONDITION_DIFFERENCE_PATH = (
    OUTPUT_DIR
    / "03_condition_difference_audit.csv"
)


rq1_original = (
    rq1_adak.loc[
        rq1_adak[
            "condition"
        ] == "original"
    ]
    .copy()
)

rq2_original = (
    rq2_adak.loc[
        rq2_adak[
            "condition"
        ] == "original"
    ]
    .copy()
)


rq1_rq3_comparison = (
    rq1_original[
        [
            "family_id",
            "code_token_count",
            "comment_lines_physical",
            "mcv",
            "sfv",
            "adak_index",
        ]
    ]
    .merge(
        rq3_adak[
            [
                "family_id",
                "code_token_count",
                "comment_lines_physical",
                "mcv",
                "sfv",
                "adak_index",
            ]
        ],
        on="family_id",
        how="left",
        suffixes=(
            "_rq1",
            "_rq3",
        ),
        validate="one_to_one",
    )
)

assert len(
    rq1_rq3_comparison
) == rq1_original[
    "family_id"
].nunique()


rq2_rq1_comparison = (
    rq2_adak[
        [
            "family_id",
            "condition",
            "code_token_count",
            "comment_lines_physical",
            "mcv",
            "sfv",
            "adak_index",
        ]
    ]
    .merge(
        rq1_adak[
            [
                "family_id",
                "condition",
                "code_token_count",
                "comment_lines_physical",
                "mcv",
                "sfv",
                "adak_index",
            ]
        ],
        on=[
            "family_id",
            "condition",
        ],
        how="left",
        suffixes=(
            "_rq2",
            "_rq1",
        ),
        validate="one_to_one",
    )
)

assert len(
    rq2_rq1_comparison
) == len(rq2_adak)


comparison_rows = []

for comparison_name, comparison_frame, left_suffix, right_suffix in [
    (
        "RQ1 original versus RQ3",
        rq1_rq3_comparison,
        "rq1",
        "rq3",
    ),
    (
        "RQ2 versus RQ1",
        rq2_rq1_comparison,
        "rq2",
        "rq1",
    ),
]:
    for metric in [
        "code_token_count",
        "comment_lines_physical",
        "mcv",
        "sfv",
        "adak_index",
    ]:
        left_values = comparison_frame[
            f"{metric}_{left_suffix}"
        ]

        right_values = comparison_frame[
            f"{metric}_{right_suffix}"
        ]

        if pd.api.types.is_numeric_dtype(
            left_values
        ):
            matches = np.isclose(
                left_values,
                right_values,
                rtol=0.0,
                atol=1e-12,
                equal_nan=False,
            )
        else:
            matches = (
                left_values
                == right_values
            )

        comparison_rows.append(
            {
                "comparison": (
                    comparison_name
                ),
                "metric": metric,
                "records": len(
                    comparison_frame
                ),
                "matching_records": int(
                    matches.sum()
                ),
                "mismatching_records": int(
                    (~matches).sum()
                ),
                "matching_percent": round(
                    100
                    * matches.mean(),
                    2,
                ),
            }
        )


cross_dataset_audit = pd.DataFrame(
    comparison_rows
)

assert (
    cross_dataset_audit[
        "mismatching_records"
    ]
    == 0
).all()


formula_audit_frames = []

for dataset_name, dataset in {
    "RQ1": rq1_adak,
    "RQ2": rq2_adak,
    "RQ3": rq3_adak,
}.items():
    audit = dataset[
        [
            "family_id",
            "language",
            "condition",
            "comment_lines_physical",
            "code_token_count",
            "mcv",
            "sfv",
            "adak_index",
        ]
    ].copy()

    audit.insert(
        0,
        "dataset",
        dataset_name,
    )

    audit[
        "comment_to_code_token_ratio"
    ] = (
        audit[
            "comment_lines_physical"
        ]
        / audit[
            "code_token_count"
        ]
    )

    audit[
        "adak_simplified_formula"
    ] = (
        (
            100
            * (
                0.8
                / 0.3
            )
            * audit[
                "comment_to_code_token_ratio"
            ]
        )
        - 100
    )

    audit[
        "formula_absolute_difference"
    ] = (
        audit[
            "adak_index"
        ]
        - audit[
            "adak_simplified_formula"
        ]
    ).abs()

    formula_audit_frames.append(
        audit
    )


formula_audit = pd.concat(
    formula_audit_frames,
    ignore_index=True,
)

assert formula_audit[
    "formula_absolute_difference"
].max() < 1e-10


formula_summary = (
    formula_audit
    .groupby(
        "dataset",
        as_index=False,
    )
    .agg(
        records=(
            "family_id",
            "count",
        ),
        maximum_formula_difference=(
            "formula_absolute_difference",
            "max",
        ),
        minimum_comment_to_code_ratio=(
            "comment_to_code_token_ratio",
            "min",
        ),
        median_comment_to_code_ratio=(
            "comment_to_code_token_ratio",
            "median",
        ),
        maximum_comment_to_code_ratio=(
            "comment_to_code_token_ratio",
            "max",
        ),
    )
)

for column in [
    "maximum_formula_difference",
    "minimum_comment_to_code_ratio",
    "median_comment_to_code_ratio",
    "maximum_comment_to_code_ratio",
]:
    formula_summary[
        column
    ] = formula_summary[
        column
    ].round(12)


range_audit = (
    rq3_adak.loc[
        (
            rq3_adak[
                "adak_index"
            ] < -100
        )
        | (
            rq3_adak[
                "adak_index"
            ] > 100
        ),
        [
            "sample_id",
            "family_id",
            "language",
            "repository",
            "file_path",
            "function_name",
            "source_url",
            "code_token_count",
            "comment_lines_physical",
            "mcv",
            "sfv",
            "adak_index",
        ],
    ]
    .copy()
)

range_audit[
    "comment_to_code_token_ratio"
] = (
    range_audit[
        "comment_lines_physical"
    ]
    / range_audit[
        "code_token_count"
    ]
)

range_audit[
    "range_violation"
] = np.where(
    range_audit[
        "adak_index"
    ] > 100,
    "above_positive_100",
    "below_negative_100",
)

range_audit = range_audit.sort_values(
    "adak_index",
    ascending=False,
).reset_index(drop=True)


range_summary = (
    rq3_adak
    .groupby(
        "language",
        as_index=False,
    )
    .agg(
        records=(
            "family_id",
            "count",
        ),
        below_negative_100=(
            "adak_below_stated_lower_bound",
            "sum",
        ),
        above_positive_100=(
            "adak_above_stated_upper_bound",
            "sum",
        ),
        minimum_adak=(
            "adak_index",
            "min",
        ),
        maximum_adak=(
            "adak_index",
            "max",
        ),
    )
)

range_summary[
    "outside_stated_range"
] = (
    range_summary[
        "below_negative_100"
    ]
    + range_summary[
        "above_positive_100"
    ]
)

range_summary[
    "outside_stated_range_percent"
] = (
    100
    * range_summary[
        "outside_stated_range"
    ]
    / range_summary[
        "records"
    ]
).round(4)


condition_difference_rows = []

for dataset_name, dataset in {
    "RQ1": rq1_adak,
    "RQ2": rq2_adak,
}.items():
    condition_values = dataset.pivot(
        index="family_id",
        columns="condition",
        values="adak_index",
    )

    condition_values[
        "original_minus_topic_swap"
    ] = (
        condition_values[
            "original"
        ]
        - condition_values[
            "topic_swap"
        ]
    )

    condition_values[
        "original_minus_grammar_shuffle"
    ] = (
        condition_values[
            "original"
        ]
        - condition_values[
            "grammar_shuffle"
        ]
    )

    condition_values[
        "topic_swap_minus_grammar_shuffle"
    ] = (
        condition_values[
            "topic_swap"
        ]
        - condition_values[
            "grammar_shuffle"
        ]
    )

    for comparison_column in [
        "original_minus_topic_swap",
        "original_minus_grammar_shuffle",
        "topic_swap_minus_grammar_shuffle",
    ]:
        differences = condition_values[
            comparison_column
        ]

        condition_difference_rows.append(
            {
                "dataset": dataset_name,
                "comparison": (
                    comparison_column
                ),
                "families": len(
                    differences
                ),
                "nonzero_differences": int(
                        (
                            ~np.isclose(
                                differences,
                                0.0,
                                rtol=0.0,
                                atol=1e-12,
                            )
                        ).sum()
                    ),
                    "minimum_difference": float(
                    differences.min()
                ),
                "median_difference": float(
                    differences.median()
                ),
                "maximum_difference": float(
                    differences.max()
                ),
            }
        )


condition_difference_audit = pd.DataFrame(
    condition_difference_rows
)

assert (
    condition_difference_audit[
        "nonzero_differences"
    ]
    == 0
).all()


cross_dataset_audit.to_csv(
    CROSS_DATASET_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

formula_audit.to_csv(
    FORMULA_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

range_audit.to_csv(
    RANGE_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

range_summary.to_csv(
    RANGE_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)

condition_difference_audit.to_csv(
    CONDITION_DIFFERENCE_PATH,
    index=False,
    encoding="utf-8",
)


print("Cross-dataset consistency")
display(cross_dataset_audit)

print("\nFormula equivalence summary")
display(formula_summary)

print("\nCondition-difference audit")
display(condition_difference_audit)

print("\nStated-range violation summary")
display(range_summary)

print("\nRecords outside the stated range")
display(range_audit)

Cross-dataset consistency


,comparison,metric,records,matching_records,mismatching_records,matching_percent
0,RQ1 original versus RQ3,code_token_count,6970,6970,0,100.0
1,RQ1 original versus RQ3,comment_lines_physical,6970,6970,0,100.0
2,RQ1 original versus RQ3,mcv,6970,6970,0,100.0
3,RQ1 original versus RQ3,sfv,6970,6970,0,100.0
4,RQ1 original versus RQ3,adak_index,6970,6970,0,100.0
5,RQ2 versus RQ1,code_token_count,20196,20196,0,100.0
6,RQ2 versus RQ1,comment_lines_physical,20196,20196,0,100.0
7,RQ2 versus RQ1,mcv,20196,20196,0,100.0
8,RQ2 versus RQ1,sfv,20196,20196,0,100.0
9,RQ2 versus RQ1,adak_index,20196,20196,0,100.0



Formula equivalence summary


,dataset,records,maximum_formula_difference,minimum_comment_to_code_ratio,median_comment_to_code_ratio,maximum_comment_to_code_ratio
0,RQ1,20910,0.0,0.002717,0.038835,1.076923
1,RQ2,20196,0.0,0.002717,0.039216,1.076923
2,RQ3,7500,0.0,0.002717,0.038462,1.076923



Condition-difference audit


,dataset,comparison,families,nonzero_differences,minimum_difference,median_difference,maximum_difference
0,RQ1,original_minus_topic_swap,6970,0,0.0,0.0,0.0
1,RQ1,original_minus_grammar_shuffle,6970,0,0.0,0.0,0.0
2,RQ1,topic_swap_minus_grammar_shuffle,6970,0,0.0,0.0,0.0
3,RQ2,original_minus_topic_swap,6732,0,0.0,0.0,0.0
4,RQ2,original_minus_grammar_shuffle,6732,0,0.0,0.0,0.0
5,RQ2,topic_swap_minus_grammar_shuffle,6732,0,0.0,0.0,0.0



Stated-range violation summary


,language,records,below_negative_100,above_positive_100,minimum_adak,maximum_adak,outside_stated_range,outside_stated_range_percent
0,go,1250,0,0,-99.257196,84.615385,0,0.00
1,java,1250,0,3,-99.275362,173.873874,3,0.24
2,javascript,1250,0,0,-99.089875,72.549020,0,0.00
3,php,1250,0,0,-98.962387,49.593496,0,0.00
4,python,1250,0,0,-99.099099,86.666667,0,0.00
5,ruby,1250,0,2,-98.476190,187.179487,2,0.16



Records outside the stated range


,sample_id,family_id,language,repository,file_path,function_name,source_url,code_token_count,comment_lines_physical,mcv,sfv,adak_index,comment_to_code_token_ratio,range_violation
0,RUBY-0187,86b4910d455968646fac546aacdf2f10dc938543c32d25...,ruby,emonti/rstruct,lib/rstruct.rb,Rstruct.ClassMethods.struct,https://github.com/emonti/rstruct/blob/0146955...,26,28,22.4,7.8,187.179487,1.076923,above_positive_100
1,JAVA-0219,abc6260b7ec791ea1123e7ea7d31424c6578be3c16723b...,java,eiichiro/bootleg,src/main/java/org/eiichiro/bootleg/Types.java,Types.isCoreValueType,https://github.com/eiichiro/bootleg/blob/b98a1...,37,38,30.4,11.1,173.873874,1.027027,above_positive_100
2,RUBY-1235,386621e0182cc1b8b4bcdaf06c776b60cf8480746a18f5...,ruby,ryanb/cancan,lib/cancan/ability.rb,CanCan.Ability.alias_action,https://github.com/ryanb/cancan/blob/4560928dc...,32,29,23.2,9.6,141.666667,0.906250,above_positive_100
3,JAVA-0369,a00775bb67509457b26968e2c838b66040f1e94fc923c8...,java,jcuda/jcublas,JCublasJava/src/main/java/jcuda/jcublas/JCubla...,JCublas.cublasDcopy,https://github.com/jcuda/jcublas/blob/80875235...,39,31,24.8,11.7,111.965812,0.794872,above_positive_100
4,JAVA-0963,f52299d8b43af56511d09c879a8127693577079ba543f3...,java,jcuda/jcublas,JCublasJava/src/main/java/jcuda/jcublas/JCubla...,JCublas.cublasIcamin,https://github.com/jcuda/jcublas/blob/80875235...,35,27,21.6,10.5,105.714286,0.771429,above_positive_100


In [7]:
import hashlib


FINAL_VALIDATION_PATH = (
    OUTPUT_DIR
    / "03_final_validation_summary.csv"
)

IMPLEMENTATION_MANIFEST_PATH = (
    OUTPUT_DIR
    / "03_adak_implementation_manifest.json"
)


def calculate_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


dataset_definitions = {
    "RQ1": {
        "dataframe": rq1_adak,
        "path": RQ1_ADAK_PATH,
        "expected_rows": 20910,
        "expected_families": 6970,
        "expected_conditions": {
            "original",
            "topic_swap",
            "grammar_shuffle",
        },
    },
    "RQ2": {
        "dataframe": rq2_adak,
        "path": RQ2_ADAK_PATH,
        "expected_rows": 20196,
        "expected_families": 6732,
        "expected_conditions": {
            "original",
            "topic_swap",
            "grammar_shuffle",
        },
    },
    "RQ3": {
        "dataframe": rq3_adak,
        "path": RQ3_ADAK_PATH,
        "expected_rows": 7500,
        "expected_families": 7500,
        "expected_conditions": {
            "original",
        },
    },
}


validation_rows = []

for dataset_name, details in (
    dataset_definitions.items()
):
    dataset = details["dataframe"]
    output_path = details["path"]

    assert output_path.exists()
    assert len(dataset) == details[
        "expected_rows"
    ]

    assert (
        dataset[
            "family_id"
        ].nunique()
        == details[
            "expected_families"
        ]
    )

    assert set(
        dataset[
            "condition"
        ].unique()
    ) == details[
        "expected_conditions"
    ]

    if dataset_name in {
        "RQ1",
        "RQ2",
    }:
        duplicate_records = int(
            dataset.duplicated(
                [
                    "family_id",
                    "condition",
                ]
            ).sum()
        )
    else:
        duplicate_records = int(
            dataset.duplicated(
                [
                    "family_id",
                ]
            ).sum()
        )

    missing_metric_values = int(
        dataset[
            [
                "method_count",
                "comment_lines_physical",
                "comment_lines_nonempty",
                "mcv",
                "sfv",
                "adak_index",
            ]
        ]
        .isna()
        .sum()
        .sum()
    )

    invalid_measurement_records = int(
        dataset[
            "adak_measurement_status"
        ]
        .ne("valid")
        .sum()
    )

    physical_line_mismatches = int(
        (
            ~dataset[
                "stored_comment_lines_match_physical"
            ]
        ).sum()
    )

    recalculated_adak = (
        (
            100
            * (
                dataset[
                    "comment_lines_physical"
                ]
                * 0.8
            )
            / (
                dataset[
                    "code_token_count"
                ]
                * 0.3
            )
        )
        - 100
    )

    maximum_formula_difference = float(
        (
            dataset[
                "adak_index"
            ]
            - recalculated_adak
        )
        .abs()
        .max()
    )

    assert duplicate_records == 0
    assert missing_metric_values == 0
    assert invalid_measurement_records == 0
    assert physical_line_mismatches == 0
    assert maximum_formula_difference < 1e-10

    validation_rows.append(
        {
            "dataset": dataset_name,
            "output_path": str(
                output_path
            ),
            "rows": len(dataset),
            "families": int(
                dataset[
                    "family_id"
                ].nunique()
            ),
            "languages": int(
                dataset[
                    "language"
                ].nunique()
            ),
            "conditions": int(
                dataset[
                    "condition"
                ].nunique()
            ),
            "duplicate_records": (
                duplicate_records
            ),
            "missing_metric_values": (
                missing_metric_values
            ),
            "invalid_measurement_records": (
                invalid_measurement_records
            ),
            "physical_line_mismatches": (
                physical_line_mismatches
            ),
            "maximum_formula_difference": (
                maximum_formula_difference
            ),
            "minimum_adak": float(
                dataset[
                    "adak_index"
                ].min()
            ),
            "median_adak": float(
                dataset[
                    "adak_index"
                ].median()
            ),
            "maximum_adak": float(
                dataset[
                    "adak_index"
                ].max()
            ),
            "below_negative_100": int(
                dataset[
                    "adak_below_stated_lower_bound"
                ].sum()
            ),
            "above_positive_100": int(
                dataset[
                    "adak_above_stated_upper_bound"
                ].sum()
            ),
            "file_size_bytes": int(
                output_path.stat().st_size
            ),
            "sha256": calculate_sha256(
                output_path
            ),
        }
    )


final_validation = pd.DataFrame(
    validation_rows
)

assert (
    cross_dataset_audit[
        "mismatching_records"
    ]
    == 0
).all()

assert (
    condition_difference_audit[
        "nonzero_differences"
    ]
    == 0
).all()

assert (
    adak_invariance[
        "inconsistent_families"
    ]
    == 0
).all()

assert (
    formula_audit[
        "formula_absolute_difference"
    ].max()
    < 1e-10
)

rq3_range_violation_count = int(
    (
        rq3_adak[
            "adak_below_stated_lower_bound"
        ]
        | rq3_adak[
            "adak_above_stated_upper_bound"
        ]
    ).sum()
)

assert rq3_range_violation_count == len(
    range_audit
)

assert int(
    range_summary[
        "outside_stated_range"
    ].sum()
) == rq3_range_violation_count


final_validation.to_csv(
    FINAL_VALIDATION_PATH,
    index=False,
    encoding="utf-8",
)


generated_files = [
    FAMILY_METRICS_PATH,
    RQ1_ADAK_PATH,
    RQ2_ADAK_PATH,
    RQ3_ADAK_PATH,
    DATASET_SUMMARY_PATH,
    LINE_AUDIT_PATH,
    LINE_SUMMARY_PATH,
    SCHEMA_AUDIT_PATH,
    TOKEN_AUDIT_PATH,
    TOKEN_CONSISTENCY_PATH,
    ADAK_SUMMARY_PATH,
    ADAK_INVARIANCE_PATH,
    CROSS_DATASET_AUDIT_PATH,
    FORMULA_AUDIT_PATH,
    RANGE_AUDIT_PATH,
    RANGE_SUMMARY_PATH,
    CONDITION_DIFFERENCE_PATH,
    FINAL_VALIDATION_PATH,
]

for path in generated_files:
    assert Path(path).exists()


file_inventory = []

for path in generated_files:
    path = Path(path)

    file_inventory.append(
        {
            "path": str(path),
            "size_bytes": int(
                path.stat().st_size
            ),
            "sha256": calculate_sha256(
                path
            ),
        }
    )


language_counts = {}

for dataset_name, details in (
    dataset_definitions.items()
):
    dataset = details["dataframe"]

    language_counts[
        dataset_name
    ] = {
        str(language): int(count)
        for language, count in (
            dataset.groupby(
                "language"
            )["family_id"]
            .nunique()
            .sort_index()
            .items()
        )
    }


implementation_manifest = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook": (
        "03_Adak_Calculation.ipynb"
    ),
    "source_paper": {
        "title": (
            "Evaluating Source Code Comment "
            "Quality: The Need for "
            "Comprehensive Indexes in "
            "Software Development"
        ),
        "authors": [
            "M. Fatih Adak",
            "M. Alp Eren Kilic",
        ],
        "publication_year": 2024,
        "doi": (
            "10.1109/"
            "ICAMAC62387.2024.10829019"
        ),
        "published_equations": {
            "mcv": (
                "((Javadoc + Other Comments) "
                "/ Methods) * 0.8"
            ),
            "sfv": (
                "(Number of Tokens "
                "/ Methods) * 0.3"
            ),
            "index": (
                "(100 * MCV / SFV) - 100"
            ),
        },
        "paper_stated_range": [
            -100,
            100,
        ],
        "clipping_rule_provided": False,
    },
    "operational_definition": {
        "unit_of_analysis": (
            "One CodeSearchNet method or "
            "function record."
        ),
        "method_count": 1,
        "comment_measurement": (
            "Physical documentation lines "
            "using Python splitlines, with "
            "a minimum of one because the "
            "eligible sample contains "
            "non-empty documentation."
        ),
        "code_measurement": (
            "CodeSearchNet code_token_count "
            "stored in the selected sample."
        ),
        "mcv_formula": (
            "comment_lines_physical "
            "* 0.8"
        ),
        "sfv_formula": (
            "code_token_count * 0.3"
        ),
        "adak_formula": (
            "(100 * mcv / sfv) - 100"
        ),
        "score_clipping": False,
        "silent_parser_fallback": False,
        "missing_measurement_policy": (
            "Flag explicitly rather than "
            "replace with a fallback."
        ),
    },
    "source_ambiguities_recorded": {
        "blank_line_treatment_unspecified": True,
        "tokenizer_unspecified": True,
        "multilanguage_adaptation_unspecified": True,
        "parser_failure_policy_unspecified": True,
        "stated_range_conflicts_with_equation": True,
    },
    "dataset_results": {
        dataset_name: {
            "path": str(
                details["path"]
            ),
            "rows": int(
                len(
                    details[
                        "dataframe"
                    ]
                )
            ),
            "families": int(
                details[
                    "dataframe"
                ][
                    "family_id"
                ].nunique()
            ),
            "language_family_counts": (
                language_counts[
                    dataset_name
                ]
            ),
        }
        for dataset_name, details in (
            dataset_definitions.items()
        )
    },
    "validation_results": {
        "rq1_adak_invariant_across_conditions": True,
        "rq2_adak_invariant_across_conditions": True,
        "cross_dataset_mismatches": int(
            cross_dataset_audit[
                "mismatching_records"
            ].sum()
        ),
        "condition_nonzero_differences": int(
            condition_difference_audit[
                "nonzero_differences"
            ].sum()
        ),
        "maximum_formula_difference": float(
            formula_audit[
                "formula_absolute_difference"
            ].max()
        ),
        "rq3_records_below_negative_100": int(
            rq3_adak[
                "adak_below_stated_lower_bound"
            ].sum()
        ),
        "rq3_records_above_positive_100": int(
            rq3_adak[
                "adak_above_stated_upper_bound"
            ].sum()
        ),
        "rq3_total_range_violations": (
            rq3_range_violation_count
        ),
    },
    "generated_files": file_inventory,
}

with IMPLEMENTATION_MANIFEST_PATH.open(
    mode="w",
    encoding="utf-8",
) as handle:
    json.dump(
        implementation_manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


print("Final validation summary")
display(
    final_validation[
        [
            "dataset",
            "rows",
            "families",
            "languages",
            "conditions",
            "duplicate_records",
            "missing_metric_values",
            "physical_line_mismatches",
            "maximum_formula_difference",
            "minimum_adak",
            "median_adak",
            "maximum_adak",
            "below_negative_100",
            "above_positive_100",
        ]
    ]
)

print("\nImplementation manifest")
print(IMPLEMENTATION_MANIFEST_PATH)

print("\nFinal validation file")
print(FINAL_VALIDATION_PATH)

Final validation summary


,dataset,rows,families,languages,conditions,duplicate_records,missing_metric_values,physical_line_mismatches,maximum_formula_difference,minimum_adak,median_adak,maximum_adak,below_negative_100,above_positive_100
0,RQ1,20910,6970,6,3,0,0,0,0.0,-99.275362,-89.644013,187.179487,0,3
1,RQ2,20196,6732,6,3,0,0,0,0.0,-99.275362,-89.542484,187.179487,0,3
2,RQ3,7500,7500,6,1,0,0,0,0.0,-99.275362,-89.743590,187.179487,0,5



Implementation manifest
C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\03_adak_calculation\03_adak_implementation_manifest.json

Final validation file
C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\03_adak_calculation\03_final_validation_summary.csv
